In [27]:
import pandas as pd
import numpy as np

In [28]:
df=pd.read_csv('supply_chain_data.csv')

In [29]:
df.columns

Index(['Product type', 'SKU', 'Price', 'Availability',
       'Number of products sold', 'Revenue generated', 'Customer demographics',
       'Stock levels', 'Lead times', 'Order quantities', 'Shipping times',
       'Shipping carriers', 'Shipping costs', 'Supplier name', 'Location',
       'Lead time', 'Production volumes', 'Manufacturing lead time',
       'Manufacturing costs', 'Inspection results', 'Defect rates',
       'Transportation modes', 'Routes', 'Costs'],
      dtype='str')

we dont need all the columns so we take only the columns needed for our project

In [30]:
required_columns = [
    "Product type",
    "SKU",
    "Production volumes",
    "Manufacturing costs",
    "Inspection results",
    "Defect rates",
    "Lead times",
]

In [31]:
df=df[required_columns]

In [32]:
df.columns


Index(['Product type', 'SKU', 'Production volumes', 'Manufacturing costs',
       'Inspection results', 'Defect rates', 'Lead times'],
      dtype='str')

now we begin to check for null values ; if there are any then we have to fill those for better dataset

In [33]:
df.isnull().sum()

Product type           0
SKU                    0
Production volumes     0
Manufacturing costs    0
Inspection results     0
Defect rates           0
Lead times             0
dtype: int64

there are no null values in our dataset

**EXECUTING LEAN PRODUCTION CALCULATIONS**

Metric A: Calculate Scrap / Defective Volume -
Formulating: Defect Rate is a percentage, convert it to a decimal multiplier

In [34]:
df["defective_volume"] = df["Production volumes"] * (df["Defect rates"] / 100)

Metric B: Calculate Cost of Poor Quality (COPQ)-
This represents the direct financial waste incurred due to scrapped items

In [35]:
df["cost_of_poor_quality"] = (df["defective_volume"] * df["Manufacturing costs"])

Metric C: Yield Efficiency Rate (%)-Calculates what percentage of production actually passed inspection successfully

In [36]:
df["yield_efficiency"] = 100 - df["Defect rates"]

Metric D: Dynamic Risk Stratification using NumPy- Industrial rule: Flag runs with high defect rates or failed inspections as Critical risk zones

In [37]:
conditions = [
    (df["Defect rates"] >= 3.0) & (df["Inspection results"] == "Fail"),
    (df["Defect rates"] >= 1.5) & (df["Defect rates"] < 3.0),
    (df["Defect rates"] < 1.5) & (df["Inspection results"] == "Pass"),
]
choices = ["Critical - Action Required", "Warning - Monitor Line", "Optimal"]

In [38]:
df["process_risk_status"] = np.select(conditions, choices, default="Unclassified")

**ROOT-CAUSE CORRELATION ANALYSIS**

Run correlation matrix to check if high production speeds (Lead Times) cause quality spikes

In [42]:
correlation_matrix = df[
    ["Production volumes", "Defect rates", "Manufacturing costs", "Lead times"]
].corr()

In [45]:
print("Quality Correlation Matrix:")
print(correlation_matrix.round(3), "\n")

Quality Correlation Matrix:
                     Production volumes  Defect rates  Manufacturing costs  \
Production volumes                1.000         0.119                0.052   
Defect rates                      0.119         1.000               -0.008   
Manufacturing costs               0.052        -0.008                1.000   
Lead times                       -0.145         0.016               -0.024   

                     Lead times  
Production volumes       -0.145  
Defect rates              0.016  
Manufacturing costs      -0.024  
Lead times                1.000  


**AGGREGATING SUMMARY STATS FOR PARETO & EXECUTIVE VIEW**

Group data by Product Type to isolate where the financial leaks are happening

In [46]:
df.columns

Index(['Product type', 'SKU', 'Production volumes', 'Manufacturing costs',
       'Inspection results', 'Defect rates', 'Lead times', 'defective_volume',
       'cost_of_poor_quality', 'yield_efficiency', 'process_risk_status'],
      dtype='str')

In [47]:
executive_summary = (
    df.groupby("Product type")
    .agg(
        total_produced=("Production volumes", "sum"),
        total_defective=("defective_volume", "sum"),
        avg_defect_rate=("Defect rates", "mean"),
        total_copq_losses=("cost_of_poor_quality", "sum"),
        avg_lead_time=("Lead times", "mean"),
    )
    .reset_index()
)


Sort by financial loss

In [48]:
executive_summary = executive_summary.sort_values(
    by="total_copq_losses", ascending=False
)
print(executive_summary.round(2), "\n")

  Product type  total_produced  total_defective  avg_defect_rate  \
2     skincare           24366           583.64             2.33   
1     haircare           19957           536.17             2.48   
0    cosmetics           12461           218.48             1.92   

   total_copq_losses  avg_lead_time  
2           28284.87          16.70  
1           26335.89          15.53  
0            7697.51          15.38   



In [49]:
df.to_csv("powerbi_supply_chain_master.csv", index=False)

In [50]:
executive_summary.to_csv("powerbi_quality_summary.csv", index=False)

**END OF TASK**